# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook loads and explores the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

**Dataset Description:**

The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

### Dataset Source
The dataset's Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, **record sets** describe logical tables of records (e.g. tabular sheets). Each record set contains one or more fields, and each field is described by an `@id`.

Let's list all record sets, their `@id`, and the `@id` of their fields.

In [ ]:
# List all record sets and their fields/columns by @id
print("Available Record Sets and Fields:")
record_sets = list(dataset.record_sets)

for record_set in record_sets:
    print(f"- Record Set Name: {record_set.name}")
    print(f"  @id: {record_set['@id']}")
    print("  Fields (columns):")
    for field in record_set.fields:
        field_id = field['@id'] if '@id' in field else None
        print(f"    - {field_id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. 

Use the record set and field `@id`s from above to select record sets for loading and analysis.

**Below, we will select the main survey record set and load it into a DataFrame using `mlcroissant`.**

In [ ]:
# Identify the available record set @ids from the listing above and choose the main one(s)
# For this dataset, often the main sheet is detected as 'cr:RecordSet/main' or similar. Let's display all and select.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"All available record set IDs: {record_set_ids}\n")

# As an example, load the first record set
dataframes = dict()
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records. Columns:\n{list(df.columns)}\n")
    dataframes[record_set_id] = df

# Let's choose the first record set for EDA/analysis
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Here, we'll apply common EDA operations using the DataFrame loaded from the chosen record set.

- **Numeric field selection:** We'll choose a numeric field (e.g. GAD-7, PHQ-9 score) by its `@id`.
- **Filtering:** We'll filter respondents by GAD-7 score above a threshold.
- **Normalization:** We'll standardize the field values.
- **Grouping:** Grouping by an attribute, e.g. `level_of_education` if present, or `gender`.

In [ ]:
### Example field selection (replace with appropriate @id from above listing, e.g. 'gad7_score')
# Find a likely column for GAD-7 or PHQ-9 score from the columns listing.
numeric_field = None
for c in df.columns:
    if "gad7" in c.lower() or "phq9" in c.lower() or "psq" in c.lower():
        numeric_field = c
        break
if not numeric_field:
    # Choose the first numeric-looking column
    for c in df.columns:
        if df[c].dtype in [int, float] or df[c].dtype.name.startswith('float'):
            numeric_field = c
            break
print(f"Selected numeric field for analysis: {numeric_field}")

# Drop missing values
eda_df = df.dropna(subset=[numeric_field])

# Set a threshold, e.g., GAD-7 > 10
threshold = 10
filtered_df = eda_df[eda_df[numeric_field] > threshold]
print(f"Number of respondents with {numeric_field} > {threshold}: {len(filtered_df)}")
print(filtered_df[[numeric_field]].head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another field (e.g. by education or gender if present)
possible_groups = [c for c in df.columns if ('education' in c.lower() or 'gender' in c.lower())]
group_field = possible_groups[0] if possible_groups else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nMean {numeric_field} by {group_field} (for values above threshold):")
    print(grouped_df)
else:
    print("No grouping field ('education' or 'gender') found.")

## 5. Visualization
Visualize the distribution of a key field and group differences.

We'll create a histogram of the selected score, and if possible, a boxplot by group (e.g. by gender/education).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8,4))

# Histogram for the numeric field
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Boxplot by group, if available
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()
else:
    print("No group field available for boxplot visualization.")

## 6. Conclusion

- We successfully loaded the FAIR² Mental Health Survey Dataset using the Croissant standard and `mlcroissant`.
- We explored record set and field structure using `@id`s as references for programmatic data access.
- We performed exploratory filtering and normalization using standardized pyschological assessment fields (e.g. GAD-7 score), and grouped analyses by demographic variable.
- Visualizations revealed the distribution of mental health assessment scores and, if present, comparison across demographic groups.

This notebook can be extended to explore additional analyses, such as correlation between different mental health scales, cross-tabulation of demographic attributes, or more advanced modeling.